# 00 · Setup — FinPay Lakehouse

Ejecutar **una sola vez** antes de correr cualquier pipeline o job.

Pasos:
1. Crear catálogo `fintech_finpay` y schemas
2. Crear volume `vol_landing` y subdirectorios
3. Crear grupos de prueba e incorporar usuarios
4. Asignar permisos diferenciados por rol
5. Crear tabla `silver.users` con column masking y row-level security
6. Validar entorno

## 0 · Parámetros globales

In [ ]:
CATALOG      = "fintech_finpay"
SCHEMAS      = ["default", "bronze", "silver", "gold", "observability"]
VOLUME       = "vol_landing"
VOLUME_PATH  = f"/Volumes/{CATALOG}/default/{VOLUME}"
SUBDIRS      = ["transactions", "merchants", "users", "metadata"]

# Grupos de prueba
GROUPS = ["ingenieria", "riesgo", "auditoria"]

print(f"Catálogo  : {CATALOG}")
print(f"Schemas   : {SCHEMAS}")
print(f"Volume    : {VOLUME_PATH}")

## 1 · Catálogo y schemas

In [ ]:
# Crear catálogo
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
print(f"✅ Catálogo '{CATALOG}' listo")

# Crear schemas
for schema in SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    print(f"  ✅ Schema '{schema}' listo")

## 2 · Volume de landing y subdirectorios

In [ ]:
# Crear volume dentro del schema 'default'
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.default.{VOLUME}
""")
print(f"✅ Volume '{VOLUME}' listo en {VOLUME_PATH}")

# Crear subdirectorios usando dbutils
for subdir in SUBDIRS:
    path = f"{VOLUME_PATH}/{subdir}"
    try:
        dbutils.fs.mkdirs(path)
        print(f"  ✅ Directorio '{subdir}' creado en {path}")
    except Exception as e:
        print(f"  ⚠️  '{subdir}': {e}")

## 3 · Grupos y usuarios de prueba

In [ ]:
# Crear grupos vía SCIM API del workspace
# Requiere token con permisos de admin
import requests, json

WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

def create_group(group_name):
    url = f"https://{WORKSPACE_URL}/api/2.0/preview/scim/v2/Groups"
    payload = {"schemas": ["urn:ietf:params:scim:schemas:core:2.0:Group"], "displayName": group_name}
    resp = requests.post(url, headers=HEADERS, json=payload)
    if resp.status_code in (200, 201):
        gid = resp.json().get("id")
        print(f"  ✅ Grupo '{group_name}' creado — id={gid}")
        return gid
    elif resp.status_code == 409:
        print(f"  ℹ️  Grupo '{group_name}' ya existe")
        # Obtener id del grupo existente
        r2 = requests.get(f"{url}?filter=displayName eq \"{group_name}\"", headers=HEADERS)
        resources = r2.json().get("Resources", [])
        return resources[0]["id"] if resources else None
    else:
        print(f"  ❌ Error creando '{group_name}': {resp.text}")
        return None

group_ids = {}
for g in GROUPS:
    group_ids[g] = create_group(g)

print(f"\nGrupos: {group_ids}")

## 4 · Permisos diferenciados por rol

In [ ]:
# ---------------------------------------------------------------------------
# Rol: ingenieria — acceso completo a todos los schemas
# ---------------------------------------------------------------------------
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `ingenieria`")

for schema in SCHEMAS:
    spark.sql(f"GRANT USE SCHEMA  ON SCHEMA {CATALOG}.{schema} TO `ingenieria`")
    spark.sql(f"GRANT CREATE TABLE ON SCHEMA {CATALOG}.{schema} TO `ingenieria`")
    spark.sql(f"GRANT MODIFY       ON SCHEMA {CATALOG}.{schema} TO `ingenieria`")
    spark.sql(f"GRANT SELECT       ON SCHEMA {CATALOG}.{schema} TO `ingenieria`")

print("✅ Permisos 'ingenieria' aplicados")

# ---------------------------------------------------------------------------
# Rol: riesgo — SELECT en silver y gold
# ---------------------------------------------------------------------------
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `riesgo`")

for schema in ["silver", "gold"]:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema} TO `riesgo`")
    spark.sql(f"GRANT SELECT     ON SCHEMA {CATALOG}.{schema} TO `riesgo`")

print("✅ Permisos 'riesgo' aplicados")

# ---------------------------------------------------------------------------
# Rol: auditoria — SELECT en gold y observability
# ---------------------------------------------------------------------------
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `auditoria`")

for schema in ["gold", "observability"]:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema} TO `auditoria`")
    spark.sql(f"GRANT SELECT     ON SCHEMA {CATALOG}.{schema} TO `auditoria`")

print("✅ Permisos 'auditoria' aplicados")

## 5 · Column Masking y Row-Level Security en silver.users

In [ ]:
# ---------------------------------------------------------------------------
# 5.1 · Función de masking para campos PII
#       Solo el grupo 'ingenieria' ve los valores reales.
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.silver.mask_pii_string(original STRING)
RETURN
  CASE
    WHEN is_member('ingenieria') THEN original
    ELSE '***REDACTED***'
  END
""")
print("✅ Función mask_pii_string creada")

In [ ]:
# ---------------------------------------------------------------------------
# 5.2 · Función de Row-Level Security
#       ingenieria ve todos los países.
#       riesgo ve todos los países (necesita datos para análisis).
#       auditoria no tiene acceso a silver.users (solo gold).
#       Por seguridad adicional, filtramos por grupo.
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.silver.rls_users_filter(country STRING)
RETURN
  CASE
    WHEN is_member('ingenieria') THEN TRUE
    WHEN is_member('riesgo')     THEN TRUE
    ELSE FALSE
  END
""")
print("✅ Función rls_users_filter creada")

In [ ]:
# ---------------------------------------------------------------------------
# 5.3 · Crear tabla silver.users con masking y RLS aplicados
#       Esta tabla es el destino del pipeline Silver para el archivo users_*.txt
#       El pipeline escribe aquí; la tabla ya define las políticas de seguridad.
# ---------------------------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.users (
    user_id           STRING   NOT NULL,
    full_name         STRING   MASK {CATALOG}.silver.mask_pii_string,
    document_id       STRING   MASK {CATALOG}.silver.mask_pii_string,
    email             STRING   MASK {CATALOG}.silver.mask_pii_string,
    phone             STRING   MASK {CATALOG}.silver.mask_pii_string,
    country           STRING,
    segment           STRING,
    registration_date DATE,
    -- Columnas técnicas de auditoría
    _source_file      STRING,
    _ingestion_ts     TIMESTAMP,
    _pipeline_run_id  STRING
)
USING DELTA
ROW FILTER {CATALOG}.silver.rls_users_filter ON (country)
COMMENT 'Usuarios FinPay — PII bajo column masking, RLS por grupo'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
)
""")
print("✅ Tabla silver.users creada con column masking y RLS")

## 6 · Tablas auxiliares de Bronze y Silver (estructura)

In [ ]:
# Bronze tables — creadas por el pipeline DLT, aquí solo documentamos
# Silver quarantine — tabla de cuarentena para registros rechazados
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.quarantine (
    quarantine_id     STRING  GENERATED ALWAYS AS (uuid()),
    source_name       STRING  NOT NULL  COMMENT 'Nombre del archivo/fuente de origen',
    rejection_reason  STRING  NOT NULL  COMMENT 'Motivo del rechazo',
    rejected_field    STRING            COMMENT 'Campo que causó el rechazo',
    original_record   STRING  NOT NULL  COMMENT 'Contenido original del registro como STRING',
    processed_at      TIMESTAMP         COMMENT 'Timestamp del procesamiento',
    pipeline_run_id   STRING            COMMENT 'ID del pipeline run'
)
USING DELTA
COMMENT 'Registros rechazados por reglas de calidad en Silver'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print("✅ Tabla silver.quarantine creada")

## 7 · Subir ingestion_archetypes.json al Volume

In [ ]:
import json

archetypes = [
    {
        "source_name": "transactions",
        "source_path": f"{VOLUME_PATH}/transactions/",
        "file_format": "csv",
        "delimiter": ",",
        "header": True,
        "target_table": f"{CATALOG}.bronze.transactions",
        "schema_location": f"{VOLUME_PATH}/metadata/checkpoints/transactions_schema",
        "checkpoint_path": f"{VOLUME_PATH}/metadata/checkpoints/transactions",
        "partition_by": ["transaction_date"],
        "active": True
    },
    {
        "source_name": "merchants",
        "source_path": f"{VOLUME_PATH}/merchants/",
        "file_format": "json",
        "delimiter": None,
        "header": None,
        "target_table": f"{CATALOG}.bronze.merchants",
        "schema_location": f"{VOLUME_PATH}/metadata/checkpoints/merchants_schema",
        "checkpoint_path": f"{VOLUME_PATH}/metadata/checkpoints/merchants",
        "partition_by": [],
        "active": True
    },
    {
        "source_name": "users",
        "source_path": f"{VOLUME_PATH}/users/",
        "file_format": "text",
        "delimiter": "|",
        "header": True,
        "target_table": f"{CATALOG}.bronze.users",
        "schema_location": f"{VOLUME_PATH}/metadata/checkpoints/users_schema",
        "checkpoint_path": f"{VOLUME_PATH}/metadata/checkpoints/users",
        "partition_by": [],
        "active": True
    }
]

archetype_path = f"{VOLUME_PATH}/metadata/ingestion_archetypes.json"
dbutils.fs.put(
    archetype_path,
    json.dumps(archetypes, indent=2, ensure_ascii=False),
    overwrite=True
)
print(f"✅ ingestion_archetypes.json escrito en {archetype_path}")

## 8 · Validación del entorno

In [ ]:
print("\n" + "="*60)
print(" VALIDACIÓN DE ENTORNO ")
print("="*60)

# Verificar schemas
schemas_db = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
for s in SCHEMAS:
    status = "✅" if s in schemas_db else "❌"
    print(f"  {status} Schema: {CATALOG}.{s}")

# Verificar volume
try:
    files = dbutils.fs.ls(VOLUME_PATH)
    dirs  = [f.name.rstrip('/') for f in files]
    for d in SUBDIRS:
        status = "✅" if d in dirs else "❌"
        print(f"  {status} Directorio: {VOLUME_PATH}/{d}")
except Exception as e:
    print(f"  ❌ Volume no accesible: {e}")

# Verificar tabla quarantine
try:
    spark.sql(f"DESCRIBE TABLE {CATALOG}.silver.quarantine")
    print(f"  ✅ Tabla: {CATALOG}.silver.quarantine")
except:
    print(f"  ❌ Tabla: {CATALOG}.silver.quarantine no encontrada")

# Verificar tabla users con masking
try:
    spark.sql(f"DESCRIBE TABLE {CATALOG}.silver.users")
    print(f"  ✅ Tabla: {CATALOG}.silver.users (con masking y RLS)")
except:
    print(f"  ❌ Tabla: {CATALOG}.silver.users no encontrada")

# Verificar archetypes
try:
    content = dbutils.fs.head(f"{VOLUME_PATH}/metadata/ingestion_archetypes.json")
    parsed  = json.loads(content)
    print(f"  ✅ ingestion_archetypes.json — {len(parsed)} fuentes configuradas")
except Exception as e:
    print(f"  ❌ ingestion_archetypes.json: {e}")

print("\n" + "="*60)
print(" Setup completado. Puedes ejecutar el pipeline ETL. ")
print("="*60)